# Optimización de rutas con A* bidireccional

## 01 · Exploración y preparación de los datos

**Objetivo:** preparar la red vial y describir los datos utilizados para estudiar rutas de menor tiempo entre paradas de San José.

Los tiempos representan condiciones de **flujo libre** (sin congestión vehicular observada) calculadas con longitudes y velocidades límite. Este notebook conserva la preparación y el análisis exploratorio realizados en el Avance 1.

## 1. Configuración y datos reproducibles

La lógica del modelo se encuentra en el paquete `rutas_gam`. Se utilizan archivos fijos para que los resultados puedan ser reproducidos sin repetir consultas externas.

Antes de ejecutar los notebooks se instalan las dependencias con `python -m pip install -r requirements.txt`.

In [ ]:
from pathlib import Path
import sys

for candidata in (Path.cwd(), *Path.cwd().parents):
    if (candidata / "rutas_gam").exists():
        RAIZ = candidata
        break
else:
    raise FileNotFoundError("Ejecute el notebook dentro del repositorio clonado.")

sys.path.insert(0, str(RAIZ))
DATOS_RAW = RAIZ / "data" / "raw"
DATOS_PROCESADOS = RAIZ / "data" / "processed"
FIGURAS = RAIZ / "figuras"
FIGURAS.mkdir(exist_ok=True)

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from rutas_gam import (
    agregar_tiempos, calcular_importancia, cargar_grafo, cargar_paradas,
    graficar_importancia, graficar_red_y_paradas, preparar_paradas,
    usar_nombres_visuales,
)

print("Paquetes cargados correctamente.")

Paquetes cargados correctamente.


## 2. De la red vial a un costo de viaje

La red se representa como un grafo dirigido: las intersecciones son nodos y cada tramo transitable es un arco. Al ser dirigido, un arco $i\to j$ no implica que también exista $j\to i$; así se respetan calles de un solo sentido.

- OSM almacena en `length` la longitud del arco en metros y puede incluir `maxspeed`, que indica el **límite de velocidad legal registrado**, no la velocidad observada.
- El paquete convierte valores de `mph` a `km/h`.
- Si no hay `maxspeed`, se asigna una velocidad según la clase `highway`, con referencia en el artículo 98 de la Ley de Tránsito.

El peso de cada arco es su tiempo estimado en minutos:

$$t_{ij}=\frac{d_{ij}}{v_{ij}}\left(\frac{60\ \text{min}}{1000\ \text{m}}\right)=\frac{d_{ij}}{v_{ij}}(0.06).$$

A* minimiza la suma de estos pesos, no la distancia geométrica.

In [ ]:
grafo_crudo = cargar_grafo(DATOS_RAW / "red_san_jose.graphml")
grafo = agregar_tiempos(grafo_crudo)
print(f"Nodos viales: {grafo.number_of_nodes():,}")
print(f"Arcos dirigidos: {grafo.number_of_edges():,}")

fuentes = pd.Series(
    [datos["speed_source"] for _, _, _, datos in grafo.edges(keys=True, data=True)]
).value_counts().rename_axis("Fuente de velocidad").reset_index(name="Arcos")
fuentes["Porcentaje %"] = fuentes["Arcos"] / fuentes["Arcos"].sum() * 100
display(fuentes.round(1))

Nodos viales: 5,975
Arcos dirigidos: 14,452


,Fuente de velocidad,Arcos,Porcentaje %
0,Imputada,12291,85.0
1,OSM,2161,15.0


In [ ]:
velocidades_por_fuente = {
    fuente: [float(datos["speed_kph"]) for _, _, _, datos in grafo.edges(keys=True, data=True)
             if datos["speed_source"] == fuente]
    for fuente in ["Imputada", "OSM"]
}
fig, ax = plt.subplots(figsize=(9, 5))
for fuente, color in [("Imputada", "#2563eb"), ("OSM", "#d97706")]:
    ax.hist(velocidades_por_fuente[fuente], bins=range(0, 131, 10), density=True,
            alpha=0.45, color=color, edgecolor="white", label=fuente)
ax.set_title("Distribución de velocidades por fuente", fontweight="bold")
ax.set_xlabel("Velocidad (km/h)")
ax.set_ylabel("Densidad")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", color="#e5e7eb", linewidth=0.8)
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(FIGURAS / "01_velocidades_por_fuente.png", dpi=180, bbox_inches="tight")
plt.show()

C:\Users\jason\AppData\Local\Temp\ipykernel_31260\1091831542.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Lectura del gráfico:** las velocidades imputadas se concentran en los valores asignados por clase vial, mientras que las registradas por OSM presentan mayor dispersión. La proporción de valores imputados debe considerarse al interpretar cualquier tiempo calculado.

## 3. ¿Cómo se eligen las paradas importantes?

Para cada parada candidata de OSM se reúnen dos señales:

1. **Densidad poblacional ($DP_i$):** densidad del distrito tomada de las estimaciones distritales 2022 del INEC.
2. **Destinos estratégicos ($DE_i$):** hospitales, clínicas, centros educativos, municipalidades, tribunales y mercados registrados en OSM dentro de 500 metros. Este radio aproxima una distancia caminable.

Como ambas variables usan escalas diferentes, primero se aplica normalización mínimo–máximo sobre todas las paradas candidatas:

$$X_i^{norm}=\frac{X_i-X_{min}}{X_{max}-X_{min}}$$

Así cada señal queda entre 0 y 1. Luego se calcula:

$$w_i=0.6\,DP_i^{norm}+0.4\,DE_i^{norm}$$


El 60 % da mayor importancia a la población potencialmente atendida y el 40 % reconoce la atracción de destinos esenciales. El generador ordena todas las candidatas por $w_i$ y guarda las 20 mayores. Este peso solo selecciona paradas: **no se suma al tiempo de los arcos ni modifica la ruta encontrada por A***.

In [ ]:
paradas = cargar_paradas(DATOS_PROCESADOS / "paradas_importantes.csv")
paradas = preparar_paradas(grafo, calcular_importancia(paradas))
paradas_presentacion = usar_nombres_visuales(paradas)
display(paradas_presentacion[[
    "Nombre", "Distrito", "Densidad poblacional",
    "Destinos estratégicos", "Índice de importancia"
]].round(3))
assert len(paradas) >= 20

,Nombre,Distrito,Densidad poblacional,Destinos estratégicos,Índice de importancia
0,Parada de Autobus.,León XIII,23915.278,3,0.633
1,Terminal León XIII,León XIII,23915.278,0,0.600
2,Torre Mercedes,Hospital,7716.566,31,0.504
3,Ruta Sabana Cementerio,Hospital,7716.566,31,0.504
4,Ruta Cementerio Sabana,Hospital,7716.566,31,0.504
5,Ruta Cementerio Sabana,Hospital,7716.566,31,0.504
6,Hospital de Niños - Paseo Colón,Hospital,7716.566,31,0.504
7,Torre Mercedes,Hospital,7716.566,31,0.504
8,Ruta Sabana Cementerio,Hospital,7716.566,30,0.493
9,Paseo Colon,Hospital,7716.566,30,0.493


In [ ]:
graficar_red_y_paradas(grafo, paradas, FIGURAS / "02_red_paradas_importantes.png")
plt.show()

C:\Users\jason\AppData\Local\Temp\ipykernel_31260\1532138572.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Lectura del mapa:** cada punto es una parada y su tamaño y color representan $w_i$. Un punto destacado puede deberse a una densidad distrital alta, a muchos destinos cercanos o a ambas condiciones. La concentración espacial también revela una limitación: al escoger únicamente las 20 puntuaciones mayores, varias paradas quedan agrupadas en el centro urbano.

> En una futura entrega se debe tomar en cuenta incorporar un mecanismo que seleccione paradas menos agrupadas.

In [ ]:
graficar_importancia(paradas, FIGURAS / "03_importancia_paradas.png")
plt.show()

C:\Users\jason\AppData\Local\Temp\ipykernel_31260\3788034661.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Lectura del gráfico:** una barra más larga indica mayor prioridad para considerar la parada como origen o destino. Los empates son esperables cuando varias plataformas pertenecen al mismo distrito y comparten los mismos destinos dentro de 500 metros. El índice no afirma cuántos pasajeros utiliza cada parada; funciona como una aproximación mientras no se disponga de demanda observada.